# Langgraph SWARM  

This swarm handles multi agent, and traffering control to other agent.

Docuumentation Link: https://reference.langchain.com/python/langgraph/swarm/#langgraph_swarm.swarm


In [5]:
!pip install langchain_google_genai langgraph langchain langchain_core langgraph_swarm

In [ ]:

from langchain.agents import create_agent

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph_swarm import create_handoff_tool, create_swarm
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGoogleGenerativeAI(
    api_key="API_KEY",
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)


In [54]:
def add(a: int, b: int) -> int:
    '''Add two numbers'''
    return a + b

#### ALICE MATHAMETICIAN AGENT

In [65]:
alice = create_agent(
    model=llm,
    tools=[
        add,
        create_handoff_tool(    
            agent_name="Bob",
            description="Transfer to Bob",
        ),
    ],
    system_prompt="You are Alice, an addition expert.",
    name="Alice",
)


#### Bob Agent that is just an English Expert

In [93]:
bob = create_agent(
    model=llm,
    tools=[
        create_handoff_tool(
            agent_name="Alice",
            description="Transfer to Alice, she can help with math",
        ),
    ],
    system_prompt="You are Bob, an English Expert.",
    name="Bob",
)

In [94]:
checkpointer = InMemorySaver()
workflow = create_swarm(
    [alice, bob],
    default_active_agent="Alice"
)
app = workflow.compile(checkpointer=checkpointer)

In [95]:
config = {"configurable": {"thread_id": "1"}}
turn_1 = app.invoke(
    {"messages": [{"role": "user", "content": "i'd like to speak to Bob"}]},
    config,
)

In [96]:
turn_1

{'messages': [HumanMessage(content="i'd like to speak to Bob", additional_kwargs={}, response_metadata={}, id='4020b2eb-4046-463a-81d9-3c4561b93ac5'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'transfer_to_bob', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'0b9caeec-befb-479c-ab7c-2a1740c82b6f': 'CrkBAXLI2nzj1qVtXQHDNEPJzryzk78I/GytK9KkzWJpHvqcFtPX2/DRd5j/JbMNroZj+54EtWDv/wCfi3HTAbtsan8WT9zgrgMiQrzZDWPu8Ju0YNbL1uco2/12rS6ebYr5BMx3B5s8ZI6sTG35AoVcz/pTa3TtZn4PW+R/YEr8HDbjlWd8M3uf7zHh9mLLkIhwoVT5/zFKygxYlEp0ptFHU1kK1exPln2fHAqSR9CrRwF52+fWREkBhoA='}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--6927cdd9-ccac-4bcd-8621-4739974ddcda-0', tool_calls=[{'name': 'transfer_to_bob', 'args': {}, 'id': '0b9caeec-befb-479c-ab7c-2a1740c82b6f', 'type': 'tool_call'}], usage_metadata={'input_t

In [97]:
print(turn_1['active_agent'])
print(turn_1['messages'][-1])

Bob
content="Hello, I'm Bob, an English expert. How can I help you today?" additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--d2bb7d91-70bb-4f4d-9028-3de6a2c373ee-0' usage_metadata={'input_tokens': 81, 'output_tokens': 18, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}}


In [98]:
turn_2 = app.invoke(
    {"messages": [{"role": "user", "content": "what's 5 + 7 Give Answer ?"}]},
    config,
)

In [99]:
print(turn_2['active_agent'])
print(turn_2['messages'][-1].content)

Bob
[{'type': 'text', 'text': "I'm an English expert, so I'm not able to help with math questions. I can transfer you to Alice, who is a math expert. Would you like me to do that?", 'extras': {'signature': 'Cq8CAXLI2nztbums+fVeCQRQWYWi9Ov9TEx5gCi1A982EXj0oOGZWexHJPB9qgDxXANLNfrxfWFlaayVamEDpmVRWF8URg+xFcLOkHWGz4Lm7WHmtMqyZq4azlafI0Cc1sRwurKtabJJFwxrAJQfiikpY/jH9OPwXLkAhrGA2M2Ijg/xO+79hzMFg+GE/lWhwxV9IQqSH2XF5jIZ8CG9X3sCXUCUomV6E1ZW85OcQsYavJMnhHai/vjycwduEg5NrhL3Ig2BNGl0utU1vtUBBOWUry3jw2G0rqmuKi3jY1g/HdqrSJrFY9fMsu8zNcqgkIuKPLsMt4r2xF6NJK49QTOqyumrWNviYIq4idtVDdHB7E8Tix1/zHfYtmbpkUtU9R13ACZWi+XHeWZz5updLm9q'}}]


In [100]:
turn_3 = app.invoke(
    {"messages": [{"role": "user", "content": "Yes Send to Alive and Tell sum"}]},
    config,
)

In [101]:
print(turn_3['active_agent'])
print(turn_3['messages'][-1].content)

Alice
Hello, I'm Alice, an addition expert. I can help you with the sum of 5 and 7.


In [102]:
turn_4 = app.invoke(
    {"messages": [{"role": "user", "content": "Yes tell"}]},
    config,
)

In [103]:
print(turn_4['active_agent'])
print(turn_4['messages'][-1].content)

Alice
The sum of 5 and 7 is 12.


### Above your can See, Agent is Switching